In [13]:
%pip install -q fomo-edge-ai kagglehub

Note: you may need to restart the kernel to use updated packages.


In [15]:
!curl -L https://huggingface.co/fomo-edge-ai/FOMO/resolve/main/weights/FOMOs.pt -o weights/FOMOs.pt -C -

** Resuming transfer from byte position 101539
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   940  100   940    0     0   5007      0 --:--:-- --:--:-- --:--:--  5026
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0


In [19]:
from fomo import FOMO
import torch
from tools.mtid_split_yolo import split_dataset
from pathlib import Path
import kagglehub

In [20]:
DATASET_PATH = Path.cwd() / "dataset"
DATASET_NAME = "andreasmoegelmose/multiview-traffic-intersection-dataset"
EPOCHS = 100
IMAGE_SIZE = 96
BATCH_SIZE = 16
dataset_path = kagglehub.dataset_download(DATASET_NAME, output_dir=str(DATASET_PATH))
yaml_path = split_dataset(dataset_path)

deduplicated 4 content-identical image(s)
total: 5772 images across train, val, test (train: 4618, val: 577, test: 577)
wrote 5772 YOLO label files beside images
wrote /Users/luis/Proyectos/gdl-atsc-anti-spillback/dataset/train.list.txt (4618 images)
wrote /Users/luis/Proyectos/gdl-atsc-anti-spillback/dataset/val.list.txt (577 images)
wrote /Users/luis/Proyectos/gdl-atsc-anti-spillback/dataset/test.list.txt (577 images)
wrote /Users/luis/Proyectos/gdl-atsc-anti-spillback/dataset/mtid.yaml


In [21]:
device = "cpu"
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"

model = FOMO(model_path="weights/FOMOs.pt", size="s", nb_classes=1, device=device)

In [22]:
results = model.train(
    allow_experimental=True,
    data=yaml_path,
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    lr0=3e-4,
    eval_interval=1,
    workers=0,
    device=device,
    project="MyProyect",
    name="GDL-ATSC-FOMO",
    exist_ok=True,
    patience=0,
)

2026-09-01 18:02:11 | INFO     | Using device: mps
2026-09-01 18:02:11 | INFO     | Setting up training...
2026-09-01 18:02:11 | INFO     | FOMOLoss: nc=1, fg_weight=100.0
2026-09-01 18:02:11 | INFO     | Dataset nc=4 differs from model nc=1 — rebuilding head.
2026-09-01 18:02:11 | INFO     | FOMOLoss rebuilt with resolved dataset nc=4
2026-09-01 18:02:11 | INFO     | FOMO training dataset: 4618 images
2026-09-01 18:02:11 | INFO     | Grid size: 12×12 (imgsz=96, downsample=8)
2026-09-01 18:02:11 | INFO     | Iterations per epoch: 289 (batch_per_rank=16, world_size=1)
2026-09-01 18:02:11 | INFO     | Optimizer: adam
2026-09-01 18:02:11 | INFO     |   - pg0 (BN): 19 params
2026-09-01 18:02:11 | INFO     |   - pg1 (Conv, wd=0.0): 20 params
2026-09-01 18:02:11 | INFO     |   - pg2 (Bias): 20 params
2026-09-01 18:02:11 | INFO     | Saving to: MyProyect/GDL-ATSC-FOMO
2026-09-01 18:02:11 | INFO     | Starting training for 100 epochs
2026-09-01 18:02:11 | INFO     | Model: FOMO-s
2026-09-01 18

KeyboardInterrupt: 